In [ ]:
import pandas as pd
import yfinance as yf
from datetime import date
pd.set_option('display.expand_frame_repr', False)

In [ ]:
end_date = "2025-12-31"
start_date = "2025-01-01"

tickers = ['NVDA', 'AAPL', 'AMZN', 'TSLA','MSFT']
data = yf.download(tickers, start=start_date, end=end_date, progress=False,auto_adjust=False)
print(data)

In [ ]:
df = data.rename_axis("Date").reset_index()
df = df.set_index("Date")
long_wide = df.stack(level=1).reset_index().rename(columns={"level_1": "Ticker"})
melted = long_wide.melt(id_vars=["Date", "Ticker"],value_vars=["Open", "High", "Low", "Close", "Volume","Adj Close"],var_name="Attribute",value_name="Value")
print(melted)

In [ ]:
data_pivoted = melted.pivot_table(index=['Date', 'Ticker'], columns='Attribute', values='Value', aggfunc='first')
stock_data = data_pivoted.reset_index()
print(stock_data)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
stock_data['Date'] = pd.to_datetime(stock_data['Date'])
stock_data.set_index('Date', inplace=True)
stock_data.reset_index(inplace=True)
plt.figure(figsize=(14, 7))
sns.set(style='whitegrid')
sns.lineplot(data=stock_data, x='Date', y='Adj Close', hue='Ticker', marker='o')
plt.xlabel('Date', fontsize=14)
plt.ylabel('Adjusted Close Price', fontsize=14)
plt.legend(title='Ticker', title_fontsize='13', fontsize='11')
plt.grid(True)
plt.xticks(rotation=45)
plt.show()

In [ ]:
stock_data['Daily Return'] = stock_data.groupby('Ticker')['Adj Close'].pct_change()
unique_tickers = stock_data['Ticker'].unique()

sns.set(style='whitegrid')

for ticker in unique_tickers:
    plt.figure(figsize=(14, 7))
    ticker_data = stock_data[stock_data['Ticker'] == ticker]
    sns.histplot(ticker_data['Daily Return'].dropna(), bins=50, kde=True, label=ticker, alpha=0.5)
    plt.title('Distribution of Daily Returns', fontsize=16)
    plt.xlabel('Daily Return', fontsize=14)
    plt.ylabel('Frequency', fontsize=14)
    plt.legend(title='Ticker', title_fontsize='13', fontsize='11')
    plt.grid(True)
    plt.tight_layout()

plt.show()

In [ ]:
daily_returns = stock_data.pivot_table(index='Date', columns='Ticker', values='Daily Return')
correlation_matrix = daily_returns.corr()

plt.figure(figsize=(12, 10))
sns.set(style='whitegrid')
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=.5, fmt='.2f', annot_kws={"size": 10})
plt.title('Correlation Matrix of Daily Returns', fontsize=16)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
expected_returns = daily_returns.mean() * 252 
volatility = daily_returns.std() * np.sqrt(252) 
stock_stats = pd.DataFrame({'Expected Return': expected_returns,'Volatility': volatility})
print(stock_stats)

In [ ]:
def portfolio_performance(weights, returns, cov_matrix):
    portfolio_return = np.dot(weights, returns)
    portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return portfolio_return, portfolio_volatility

In [ ]:
num_portfolios = 10000
results = np.zeros((3+len(ticker)+1, num_portfolios))
cov_matrix = daily_returns.cov() * 252

np.random.seed(456)

for i in range(num_portfolios):
    weights = np.random.random(len(unique_tickers))
    weights /= np.sum(weights)
    portfolio_return, portfolio_volatility = portfolio_performance(weights, expected_returns, cov_matrix)
    results[0,i] = portfolio_return
    results[1,i] = portfolio_volatility
    results[2,i] = portfolio_return / portfolio_volatility # Sharpe Ratio
    results[3:3+len(ticker)+1,i] = weights#[0]

labels = ['returns','volatility','sharpe']+ [f"w{i}" for i in range(1,1+len(tickers))]
res = pd.DataFrame(results.T,columns=labels)

print(res)

In [ ]:
plt.figure(figsize=(10, 7))
plt.scatter(res.volatility, res.returns, c=res.sharpe, cmap='YlGnBu', marker='o')
plt.xlabel('Volatility (Standard Deviation)')
plt.ylabel('Expected Return')
plt.colorbar(label='Sharpe Ratio')
plt.grid(True)
plt.show()

In [ ]:
print(res.iloc[np.argmax(res.sharpe),:])